# Multi-Source Knowledge Base

In practice, knowledge bases are built incrementally from multiple sources — reports, articles, notes, and documents added over time. This notebook demonstrates:

1. **Incremental indexing** — adding new sources to an existing knowledge graph
2. **Cross-source queries** — retrieving information that spans multiple documents
3. **Workspace isolation** — keeping separate knowledge bases independent

In [ ]:
import os
import time

from dotenv import load_dotenv

load_dotenv()

from neocortex import GraphRAG, QueryParam
from neocortex._llm import token_tracker

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready.")

## Source 1: Encyclopedia Articles

Three articles covering key topics in space exploration history.

In [ ]:
ARTICLES = [
    (
        "The Apollo program was a NASA spaceflight program that ran from 1961 to 1972. "
        "It successfully landed twelve astronauts on the Moon across six missions: Apollo 11 "
        "through Apollo 17 (excluding Apollo 13, which was aborted due to an in-flight "
        "malfunction). The program was conceived during the Eisenhower administration and "
        "championed by President John F. Kennedy, who in 1961 declared the national goal of "
        "landing a man on the Moon before the end of the decade. The program employed over "
        "400,000 workers and cost approximately $25.4 billion (equivalent to roughly $200 "
        "billion in 2024 dollars). Neil Armstrong and Buzz Aldrin became the first humans to "
        "walk on the Moon on July 20, 1969, during Apollo 11, while Michael Collins orbited "
        "above in the command module. The Saturn V rocket, designed by Wernher von Braun's "
        "team at Marshall Space Flight Center, was the launch vehicle for all crewed lunar "
        "missions. It remains the most powerful rocket ever successfully flown.",
        {"source": "encyclopedia", "topic": "Apollo Program", "category": "space history"}
    ),
    (
        "The International Space Station (ISS) is a modular space station in low Earth orbit. "
        "It is a multinational collaborative project involving five participating space "
        "agencies: NASA (United States), Roscosmos (Russia), JAXA (Japan), ESA (Europe), and "
        "CSA (Canada). The station was launched in 1998 and has been continuously occupied "
        "since November 2000. It orbits at an altitude of approximately 420 km and travels "
        "at 27,600 km/h, completing one orbit every 90 minutes. The ISS serves as a "
        "microgravity research laboratory for biology, physics, astronomy, and other fields. "
        "Over 270 astronauts from 21 countries have visited the station. The ISS is roughly "
        "the size of a football field, measuring 109 meters from end to end, and weighs "
        "approximately 420,000 kg. It was assembled piece by piece in orbit over a decade, "
        "with the first module, Zarya, launched by Russia, followed by the Unity node from "
        "the United States. The station is expected to operate until 2030, after which it "
        "will be deorbited.",
        {"source": "encyclopedia", "topic": "International Space Station", "category": "space infrastructure"}
    ),
    (
        "SpaceX, founded by Elon Musk in 2002, revolutionized spaceflight by developing "
        "reusable rockets. The Falcon 9 rocket, first launched in 2010, became the first "
        "orbital-class rocket to successfully land its first stage booster for reuse. By 2024, "
        "Falcon 9 had completed over 300 missions with a success rate exceeding 99%. SpaceX "
        "also developed the Dragon spacecraft, which in 2020 became the first commercial "
        "vehicle to carry astronauts to the ISS as part of NASA's Commercial Crew Program. "
        "The company's Starship program aims to create a fully reusable super-heavy launch "
        "vehicle capable of carrying 150 tonnes to low Earth orbit. Starship is central to "
        "NASA's Artemis program, which plans to return humans to the Moon. SpaceX operates "
        "from launch facilities at Kennedy Space Center in Florida, Vandenberg Space Force "
        "Base in California, and Starbase in Boca Chica, Texas. The company also runs the "
        "Starlink satellite internet constellation, with over 5,000 satellites in orbit "
        "providing broadband coverage to more than 60 countries.",
        {"source": "encyclopedia", "topic": "SpaceX", "category": "commercial space"}
    ),
]

print(f"Source 1: {len(ARTICLES)} encyclopedia articles")

## Source 2: Research Notes

Informal research notes with a different metadata schema, covering recent developments.

In [ ]:
RESEARCH_NOTES = [
    (
        "Notes on Artemis Program (updated Jan 2025):\n"
        "- Artemis I: Uncrewed test flight completed successfully in December 2022. Orion "
        "capsule orbited the Moon for 25 days.\n"
        "- Artemis II: Crewed flyby mission scheduled for 2025. Crew: Reid Wiseman (commander), "
        "Victor Glover (pilot), Christina Koch, and Jeremy Hansen (CSA).\n"
        "- Artemis III: First crewed lunar landing since Apollo 17. Will use SpaceX Starship "
        "as the Human Landing System (HLS). Target: 2026.\n"
        "- Gateway: Lunar orbital station being developed jointly by NASA and ESA. Will serve "
        "as a staging point for lunar surface missions.\n"
        "- Key difference from Apollo: Artemis will land the first woman and first person of "
        "color on the Moon. South Pole landing site chosen for water ice access.",
        {"source": "research_notes", "author": "Dr. Sarah Mitchell", "last_updated": "2025-01-15"}
    ),
    (
        "Mars Exploration Timeline:\n"
        "- Mars Pathfinder (1997): First successful rover (Sojourner) on Mars.\n"
        "- Spirit and Opportunity (2004): Twin rovers. Opportunity operated for 15 years.\n"
        "- Curiosity (2012): Car-sized rover in Gale Crater. Still operational.\n"
        "- Perseverance (2021): Landed in Jezero Crater with Ingenuity helicopter. First "
        "powered flight on another planet. Collecting samples for Mars Sample Return mission.\n"
        "- Mars Sample Return: Joint NASA/ESA mission to bring Perseverance's collected "
        "samples back to Earth. Under redesign as of 2024 due to cost overruns.\n"
        "- SpaceX has stated goal of sending uncrewed Starship to Mars by 2026 and eventually "
        "establishing a permanent human settlement.",
        {"source": "research_notes", "author": "Dr. Sarah Mitchell", "last_updated": "2025-01-20"}
    ),
]

print(f"Source 2: {len(RESEARCH_NOTES)} research notes")

## Source 3: Technical Report

A longer technical report on satellite technology.

In [ ]:
TECHNICAL_REPORT = (
    "Technical Report: The Evolution of Satellite Communications\n\n"
    "1. Introduction\n"
    "Satellite communications have undergone a dramatic transformation since the launch of "
    "Sputnik in 1957. The first communications satellite, Telstar 1, was launched in 1962 "
    "by Bell Telephone Laboratories and enabled the first live transatlantic television "
    "broadcast. Today, over 10,000 active satellites orbit Earth, providing services ranging "
    "from GPS navigation to broadband internet.\n\n"
    "2. Geostationary Satellites\n"
    "Traditional communications satellites operate in geostationary orbit (GEO) at 35,786 km "
    "altitude. Companies like Intelsat, SES, and Eutelsat have operated GEO fleets for decades. "
    "The main advantage of GEO is that satellites appear stationary relative to the ground, "
    "simplifying antenna pointing. However, the high altitude introduces 600ms round-trip "
    "latency, making GEO unsuitable for real-time applications.\n\n"
    "3. Low Earth Orbit Constellations\n"
    "The new paradigm in satellite communications is mega-constellations in low Earth orbit "
    "(LEO), at altitudes of 340-550 km. SpaceX's Starlink leads with over 5,000 satellites, "
    "followed by OneWeb (648 satellites, backed by the UK government and Bharti Global) and "
    "Amazon's Project Kuiper (planning 3,236 satellites). LEO constellations offer latencies "
    "of 20-40ms, comparable to terrestrial fiber optic networks.\n\n"
    "4. Laser Inter-Satellite Links\n"
    "Modern LEO constellations use laser inter-satellite links (ISLs) to route traffic between "
    "satellites in orbit, reducing the need for ground stations. SpaceX began deploying ISL-"
    "equipped Starlink satellites in 2021. These optical links operate at speeds up to 100 Gbps "
    "and can potentially route data faster than terrestrial fiber by taking shorter paths through "
    "the vacuum of space.\n\n"
    "5. Future Trends\n"
    "Key trends include direct-to-cell satellite service (T-Mobile and SpaceX partnership for "
    "coverage via standard smartphones), satellite-based quantum key distribution for secure "
    "communications, and increasingly autonomous satellite operations using onboard AI for "
    "collision avoidance and spectrum management."
)

REPORT_METADATA = {"source": "technical_report", "title": "Evolution of Satellite Communications", "year": "2025"}

print(f"Source 3: Technical report ({len(TECHNICAL_REPORT)} characters)")

## Incremental Indexing

We'll add each source one at a time, tracking how the graph grows with each addition.

In [ ]:
rag = GraphRAG(
    working_dir="./db/examples_multi_source",
    domain=(
        "space exploration, aerospace technology, satellite communications, "
        "NASA programs, commercial spaceflight, and planetary science"
    ),
    example_queries=(
        "What is the Artemis program?\n"
        "How do LEO satellite constellations work?\n"
        "What is the relationship between SpaceX and NASA?\n"
        "What rovers have explored Mars?"
    ),
    entity_types=["person", "organization", "spacecraft", "location", "mission", "technology"],
)

print("GraphRAG initialized.")

In [ ]:
stats = []

# Source 1: Encyclopedia articles
article_texts = [text for text, _ in ARTICLES]
article_meta = [meta for _, meta in ARTICLES]

token_tracker.reset()
start = time.perf_counter()
e, r, c = rag.insert(article_texts, metadata=article_meta)
elapsed = time.perf_counter() - start
stats.append(("Encyclopedia", e, r, c, elapsed, token_tracker.total_cost))
print(f"[1/3] Encyclopedia:    {e} entities, {r} relations, {c} chunks ({elapsed:.1f}s, ${token_tracker.total_cost:.4f})")

# Source 2: Research notes
note_texts = [text for text, _ in RESEARCH_NOTES]
note_meta = [meta for _, meta in RESEARCH_NOTES]

token_tracker.reset()
start = time.perf_counter()
e, r, c = rag.insert(note_texts, metadata=note_meta)
elapsed = time.perf_counter() - start
stats.append(("Research Notes", e, r, c, elapsed, token_tracker.total_cost))
print(f"[2/3] Research Notes:  {e} entities, {r} relations, {c} chunks ({elapsed:.1f}s, ${token_tracker.total_cost:.4f})")

# Source 3: Technical report
token_tracker.reset()
start = time.perf_counter()
e, r, c = rag.insert(TECHNICAL_REPORT, metadata=REPORT_METADATA)
elapsed = time.perf_counter() - start
stats.append(("Technical Report", e, r, c, elapsed, token_tracker.total_cost))
print(f"[3/3] Technical Report: {e} entities, {r} relations, {c} chunks ({elapsed:.1f}s, ${token_tracker.total_cost:.4f})")

print(f"\nTotal: {sum(s[1] for s in stats)} entities, {sum(s[2] for s in stats)} relations, {sum(s[3] for s in stats)} chunks")

## Cross-Source Queries

Queries that require synthesizing information from multiple sources — the graph connects entities across documents.

In [ ]:
QUESTIONS = [
    # Spans encyclopedia + research notes
    "How does the Artemis program compare to the original Apollo program?",
    # Spans encyclopedia + technical report
    "What role does SpaceX play in both crewed spaceflight and satellite communications?",
    # Spans research notes + technical report
    "What are the major upcoming milestones in space exploration and satellite technology?",
]

for question in QUESTIONS:
    token_tracker.reset()
    start = time.perf_counter()

    response = rag.query(question, params=QueryParam.balanced())

    elapsed = time.perf_counter() - start

    # Identify which sources contributed
    source_types = set()
    for chunk, _ in response.context.chunks:
        source_types.add(chunk.metadata.get("source", "unknown"))

    print(f"\nQ: {question}")
    print(f"  ({elapsed:.1f}s | ${token_tracker.total_cost:.4f} | sources: {', '.join(sorted(source_types))})")
    print(f"\nA: {response.response}")
    print("=" * 80)

## Workspace Isolation

Different `working_dir` paths create completely separate knowledge bases. They share nothing — no entities, relations, or chunks leak between workspaces.

In [ ]:
# Create a separate knowledge base about cooking
cooking_rag = GraphRAG(
    working_dir="./db/examples_isolated",
    domain="culinary arts, recipes, cooking techniques, and food science",
    example_queries=(
        "What is the Maillard reaction?\n"
        "How do you make a roux?\n"
        "What are the five French mother sauces?"
    ),
    entity_types=["ingredient", "technique", "dish", "cuisine", "person"],
)

COOKING_TEXT = """
The five French mother sauces, codified by Auguste Escoffier in the early 20th century,
form the foundation of classical French cuisine. Bechamel is made from butter, flour,
and milk — a white roux thinned with warm milk and seasoned with nutmeg. Veloute starts
with a blond roux and light stock (chicken, fish, or veal). Espagnole uses a dark roux
with brown stock, tomato puree, and a mirepoix of onion, carrot, and celery. Sauce
tomat (tomato sauce) combines tomatoes with pork, aromatics, and roux. Hollandaise is
an emulsion of egg yolks and clarified butter, flavored with lemon juice.

The Maillard reaction, named after French chemist Louis-Camille Maillard, is a chemical
reaction between amino acids and reducing sugars that occurs at temperatures above 140C
(280F). It is responsible for the browning and complex flavors of seared steak, toasted
bread, roasted coffee, and many other cooked foods. The reaction produces hundreds of
different flavor compounds and is distinct from caramelization, which involves only sugars.
""".strip()

token_tracker.reset()
start = time.perf_counter()
e, r, c = cooking_rag.insert(COOKING_TEXT)
elapsed = time.perf_counter() - start
print(f"Cooking KB indexed: {e} entities, {r} relations, {c} chunks ({elapsed:.1f}s)")

In [ ]:
# Query the cooking KB — should know about sauces
response = cooking_rag.query("What are the five mother sauces?")
print("Cooking KB:")
print(f"  Q: What are the five mother sauces?")
print(f"  A: {response.response[:300]}")
print()

# Query the space KB with a cooking question — should not know
response = rag.query("What are the five mother sauces?")
print("Space KB:")
print(f"  Q: What are the five mother sauces?")
print(f"  A: {response.response[:300]}")
print()
print("The two workspaces are completely isolated — no knowledge leaks between them.")

## Summary

- **Incremental indexing** lets you grow a knowledge base over time by calling `rag.insert()` multiple times.
- **Cross-source queries** work automatically — the graph connects entities across all indexed documents.
- **Workspace isolation** uses different `working_dir` paths to create independent knowledge bases.
- **Metadata** is preserved per-chunk, so you can always trace back to the original source.

**Next steps:**
- [06_graph_visualization.ipynb](06_graph_visualization.ipynb) — Export and visualize the knowledge graph